## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

### Experiment 1: Decision Tree with `max_depth=3` (original features)

In [8]:
# Train a Decision Tree Classifier with max_depth=3
tree_depth_3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth_3.fit(X, y)

print("Decision Tree with max_depth=3 (original features):")
print(export_text(tree_depth_3, feature_names=features))

# Evaluate Precision@50
tree_score_depth_3 = tree_depth_3.predict_proba(X)[:, 1]

hr_50 = precision_at_k(df["hand_rule_score"], y, 50)
tr_depth_3_50 = precision_at_k(tree_score_depth_3, y, 50)

print(f"\nPrecision@50:  hand rule {hr_50:.3f}   vs   tree (depth 3) {tr_depth_3_50:.3f}")

Decision Tree with max_depth=3 (original features):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0


Precision@50:  hand rule 0.680   vs   tree (depth 3) 0.720


### Experiment 2: Decision Tree with `max_depth=3` (features: `impressions_90d` dropped, `engagement_rate` added)

In [9]:
# Define new features: drop 'impressions_90d' and add 'engagement_rate'
new_features_1 = [f for f in features if f != "impressions_90d"] + ["engagement_rate"]
X_new_1 = df[new_features_1].replace([np.inf, -np.inf], np.nan).fillna(0)

# Train a Decision Tree Classifier with max_depth=3 using new features
tree_new_features_1 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_new_features_1.fit(X_new_1, y)

print("Decision Tree with max_depth=3 (new features):")
print(export_text(tree_new_features_1, feature_names=new_features_1))

# Evaluate Precision@50
tree_score_new_features_1 = tree_new_features_1.predict_proba(X_new_1)[:, 1]

tr_new_features_1_50 = precision_at_k(tree_score_new_features_1, y, 50)

print(f"\nPrecision@50:  hand rule {hr_50:.3f}   vs   tree (depth 3, new features) {tr_new_features_1_50:.3f}")

Decision Tree with max_depth=3 (new features):
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0


Precision@50:  hand rule 0.680   vs   tree (depth 3, new features) 0.740


### Experiment 3: Decision Tree with `max_depth=4` (features: `impressions_90d` dropped, `engagement_rate` added)

In [10]:
# Train a Decision Tree Classifier with max_depth=4 using new features
tree_new_features_2 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_new_features_2.fit(X_new_1, y)

print("Decision Tree with max_depth=4 (new features):")
print(export_text(tree_new_features_2, feature_names=new_features_1))

# Evaluate Precision@50
tree_score_new_features_2 = tree_new_features_2.predict_proba(X_new_1)[:, 1]

tr_new_features_2_50 = precision_at_k(tree_score_new_features_2, y, 50)

print(f"\nPrecision@50:  hand rule {hr_50:.3f}   vs   tree (depth 4, new features) {tr_new_features_2_50:.3f}")

Decision Tree with max_depth=4 (new features):
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- word_count <= 653.50
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  653.50
|   |   |   |   |--- class: 1
|   |   |--- word_count >  669.50
|   |   |   |--- days_since_last_update <= 3.50
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  3.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- word_count <= 1653.50
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  1653.50
|   |   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- content_age_days <= 172.50
|   |   |   |   |--- class: 1
|   |   |   |--- content_age_days >  172.50
|   |   |   |   |--- class: 1
|   |   |--- ctr >

### Experiment 4: Client-Holdout Validation

First, let's examine the `scripts/03_train_model.py` file to understand the client-holdout validation strategy.

In [11]:
# Display the content of 03_train_model.py
!cat scripts/03_train_model.py

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_utils import (
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    OUTPUT_DIR,
    PROCESSED_DIR,
    display_path,
    precision_at_k,
    write_json,
)


FEATURE_PATH = PROCESSED_DIR / "refresh_feature_vector.csv"
BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
PREDICTION_PATH = PROCESSED_DIR / "model_predictions.csv"
RESULT_PATH = OUTPUT_DIR / "model_results.json"
RANDOM_STATE = 42


def parse_args

### Implementing Client-Holdout Validation

In [12]:
from sklearn.model_selection import train_test_split

# Prepare features and target for the best model from previous experiments
# Re-create X_new_1 to ensure 'engagement_rate' is available and features match

# Add 'engagement_rate' to the original df if not already present
if 'engagement_rate' not in df.columns:
    df['engagement_rate'] = df['ctr'] / df['impressions_90d'] # Example definition, adjust if actual definition is different
    df['engagement_rate'] = df['engagement_rate'].replace([np.inf, -np.inf], np.nan).fillna(0)

new_features_best_model = [f for f in features if f != "impressions_90d"] + ["engagement_rate"]
X_client_holdout = df[new_features_best_model].replace([np.inf, -np.inf], np.nan).fillna(0)
y_client_holdout = df["is_declining_label"].values

# Get unique client IDs
unique_clients = df["client_id"].unique()

# Determine the number of test clients (approx 20%)
test_client_count = max(1, len(unique_clients) // 5)

# Randomly select test client IDs
np.random.seed(42) # for reproducibility
test_client_ids = np.random.choice(unique_clients, test_client_count, replace=False)

# Split data into train and test sets based on client IDs
is_test_client = df["client_id"].isin(test_client_ids)

X_train, X_test = X_client_holdout[~is_test_client], X_client_holdout[is_test_client]
y_train, y_test = y_client_holdout[~is_test_client], y_client_holdout[is_test_client]

# Store the hand rule scores for train and test sets separately	rain_hand_rule_scores = df.loc[~is_test_client, "hand_rule_score"]
test_hand_rule_scores = df.loc[is_test_client, "hand_rule_score"]

print(f"Total data points: {len(df)}")
print(f"Train data points: {len(X_train)}")
print(f"Test data points: {len(X_test)}")
print(f"Unique clients in train: {df.loc[~is_test_client, 'client_id'].nunique()}")
print(f"Unique clients in test: {df.loc[is_test_client, 'client_id'].nunique()}")
print(f"Clients selected for test: {test_client_ids.tolist()}")

Total data points: 30000
Train data points: 26619
Test data points: 3381
Unique clients in train: 26
Unique clients in test: 6
Clients selected for test: ['client_8b940be7fb', 'client_bbb965ab0c', 'client_9400f1b21c', 'client_a88a7902cb', 'client_8527a891e2', 'client_9f14025af0']


### Evaluate Best Model with Client-Holdout Split

In [13]:
# Train the Decision Tree Classifier with max_depth=3 on the training data
tree_holdout = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_holdout.fit(X_train, y_train)

# Evaluate Precision@50 on the TEST data
tree_score_test = tree_holdout.predict_proba(X_test)[:, 1]

hr_50_test = precision_at_k(test_hand_rule_scores, y_test, 50)
tr_holdout_50_test = precision_at_k(tree_score_test, y_test, 50)

print(f"\nPrecision@50 (Test Set):  hand rule {hr_50_test:.3f}   vs   tree (depth 3, new features) {tr_holdout_50_test:.3f}")

print("\nDecision Tree (Depth 3, New Features) trained on Train Set and evaluated on Test Set:")
print(export_text(tree_holdout, feature_names=new_features_best_model))


Precision@50 (Test Set):  hand rule 0.640   vs   tree (depth 3, new features) 0.600

Decision Tree (Depth 3, New Features) trained on Train Set and evaluated on Test Set:
|--- avg_position <= 0.55
|   |--- word_count <= 687.00
|   |   |--- ctr <= 0.04
|   |   |   |--- class: 0
|   |   |--- ctr >  0.04
|   |   |   |--- class: 1
|   |--- word_count >  687.00
|   |   |--- avg_position <= 0.25
|   |   |   |--- class: 0
|   |   |--- avg_position >  0.25
|   |   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.31
|   |   |   |--- class: 1
|   |   |--- ctr >  0.31
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- days_since_last_update <= 25.50
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  25.50
|   |   |   |--- class: 1

